# Ensemble from saved source-model artifacts

This notebook reloads the best tabular-notes model and the trained signal model from the two source notebooks, blends their predictions, and writes a submission file.

In [19]:
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
import torch
from feature_encoders import MaintenanceNoteTargetEncoder
from signal_model import DroneCrashClassifier

ARTIFACTS_DIR = Path("artifacts")
TABULAR_MODEL_PATH = ARTIFACTS_DIR / "tabular_model.joblib"
TABULAR_METADATA_PATH = ARTIFACTS_DIR / "tabular_model_metadata.joblib"
SIGNAL_MODEL_PATH = ARTIFACTS_DIR / "signal_model.pt"

print("Artifacts directory:", ARTIFACTS_DIR)

Artifacts directory: artifacts


In [20]:
def load_tabular_frame(path: str) -> pd.DataFrame:
    tabular = pd.read_csv(path)
    notes_path = "data/train/train_notes.csv" if "train" in path else "data/test/test_notes.csv"
    notes = pd.read_csv(notes_path)
    merged = pd.merge(tabular, notes, on="flight_id", how="inner")
    if "maintenance_note" in merged.columns:
        merged["maintenance_note"] = merged["maintenance_note"].fillna("").astype(str)
    else:
        merged["maintenance_note"] = ""
    if "failure_mode" in merged.columns:
        merged["failure_mode"] = merged["failure_mode"].fillna("none").astype(str)
    else:
        merged["failure_mode"] = "none"
    return merged


metadata = joblib.load(TABULAR_METADATA_PATH)
tabular_model = joblib.load(TABULAR_MODEL_PATH)
feature_columns = metadata["feature_columns"]

test_df = load_tabular_frame("data/test/test_tabular.csv")
X_test = test_df.reindex(columns=feature_columns, fill_value=np.nan)

tabular_test_prob = tabular_model.predict_proba(X_test)[:, 1]
print("Loaded tabular model and generated", len(tabular_test_prob), "probabilities")

Loaded tabular model and generated 7534 probabilities


In [21]:
def load_signal_array(path: str) -> np.ndarray:
    z = np.load(path)
    return z["signals"].transpose(0, 2, 1)


train_signals = load_signal_array("data/train/train_signals.npz")
test_signals = load_signal_array("data/test/test_signals.npz")

train_mean = train_signals.mean(axis=(0, 2), keepdims=True)
train_std = train_signals.std(axis=(0, 2), keepdims=True) + 1e-8

signal_model_artifact = torch.load(SIGNAL_MODEL_PATH, map_location="cpu", weights_only=False)
if isinstance(signal_model_artifact, dict):
    model = DroneCrashClassifier()
    model.load_state_dict(signal_model_artifact)
else:
    model = signal_model_artifact
model.eval()

with torch.no_grad():
    normalized_test = (test_signals - train_mean) / train_std
    signal_test_prob = model(torch.tensor(normalized_test, dtype=torch.float32)).numpy()

print("Loaded signal model and generated", len(signal_test_prob), "probabilities")

Loaded signal model and generated 7534 probabilities


In [ ]:
from sklearn.metrics import average_precision_score

ensemble_prob = 0.5 * tabular_test_prob + 0.5 * signal_test_prob
ensemble_prob = np.clip(ensemble_prob, 0.0, 1.0)

# Evaluate on the aligned training set for a quick sanity check.
train_tabular = pd.read_csv("data/train/train_tabular.csv")
train_notes = pd.read_csv("data/train/train_notes.csv")
train_merged = pd.merge(train_tabular, train_notes, on="flight_id", how="inner")
train_merged["maintenance_note"] = train_merged["maintenance_note"].fillna("").astype(str)
train_merged["failure_mode"] = train_merged["failure_mode"].fillna("none").astype(str)
train_features = train_merged.reindex(columns=feature_columns, fill_value=np.nan)
train_y = train_merged["failure_within_horizon"].astype(float)

train_tabular_prob = tabular_model.predict_proba(train_features)[:, 1]

signals_npz = np.load("data/train/train_signals.npz")
signal_ids = signals_npz["flight_id"]
signal_arrays = signals_npz["signals"].transpose(0, 2, 1)
signal_lookup = pd.DataFrame({"flight_id": signal_ids, "array_idx": np.arange(len(signal_ids))})
label_df = train_tabular[["flight_id", "failure_within_horizon"]]
aligned = signal_lookup.merge(label_df, on="flight_id", how="inner")

aligned_features = train_merged.set_index("flight_id").loc[aligned["flight_id"].values].reset_index()
aligned_tabular_prob = tabular_model.predict_proba(aligned_features.reindex(columns=feature_columns, fill_value=np.nan))[:, 1]

aligned_signals = signal_arrays[aligned["array_idx"].values]
with torch.no_grad():
    normalized_train = (aligned_signals - train_signals.mean(axis=(0, 2), keepdims=True)) / (train_signals.std(axis=(0, 2), keepdims=True) + 1e-8)
    signal_train_prob = model(torch.tensor(normalized_train, dtype=torch.float32)).numpy()

ensemble_train_prob = 0.5 * aligned_tabular_prob + 0.5 * signal_train_prob
labels = aligned["failure_within_horizon"].astype(float)

print("AUPRC tabular:", round(float(average_precision_score(labels, aligned_tabular_prob)), 6))
print("AUPRC signal:", round(float(average_precision_score(labels, signal_train_prob)), 6))
print("AUPRC ensemble:", round(float(average_precision_score(labels, ensemble_train_prob)), 6))

submission = pd.read_csv("data/sample_submission.csv")
submission["failure_within_horizon"] = ensemble_prob
submission.to_csv("data/submission_from_saved_models.csv", index=False)

print(submission.head())
print("Saved submission to data/submission_from_saved_models.csv")

KeyError: '[15872, 15873, 15874, 15875, 15876, 15877, 15878, 15879, 15880, 15881, 15882, 15883, 15884, 15885, 15886, 15887, 15888, 15889, 15890, 15891, 15892, 15893, 15894, 15895, 15896, 15897, 15898, 15899, 15900, 15901, 15902, 15903, 15904, 15905, 15906, 15907, 15908, 15909, 15910, 15911, 15912, 15913, 15914, 15915, 15916, 15917, 15918, 15919, 15920, 15921, 15922, 15923, 15924, 15925, 15926, 15927, 15928, 15929, 15930, 15931, 15932, 15933, 15934, 15935, 15936, 15937, 15938, 15939, 15940, 15941, 15942, 15943, 15944, 15945, 15946, 15947, 15948, 15949, 15950, 15951, 15952, 15953, 15954, 15955, 15956, 15957, 15958, 15959, 15960, 15961, 15962, 15963, 15964, 15965, 15966, 15967, 15968, 15969, 15970, 15971, 15972, 15973, 15974, 15975, 15976, 15977, 15978, 15979, 15980, 15981, 15982, 15983, 15984, 15985, 15986, 15987, 15988, 15989, 15990, 15991, 15992, 15993, 15994, 15995, 15996, 15997, 15998, 15999, 16000, 16001, 16002, 16003, 16004, 16005, 16006, 16007, 16008, 16009, 16010, 16011, 16012, 1